In [1]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/raw/ec2_cpu_utilization_24ae8d.csv', parse_dates=['timestamp'])
df = df.set_index('timestamp')
df = df.asfreq('5min')

tamanho_treino = int(len(df) * 0.8)
treino = df['value'].iloc[:tamanho_treino]
teste = df['value'].iloc[tamanho_treino:]

In [3]:
def termos_fourier(indice, periodo, n_harmonicos, t_offset=0):
    t = np.arange(len(indice)) + t_offset
    termos = {}
    for k in range(1, n_harmonicos + 1):
        termos[f'seno_{k}'] = np.sin(2 * np.pi * k * t / periodo)
        termos[f'cos_{k}'] = np.cos(2 * np.pi * k * t / periodo)
    return pd.DataFrame(termos, index=indice)

N_HARMONICOS = 3
exog_treino = termos_fourier(treino.index, 288, N_HARMONICOS, t_offset=0)
exog_teste = termos_fourier(teste.index, 288, N_HARMONICOS, t_offset=len(treino))

modelo = SARIMAX(treino, exog=exog_treino, order=(1, 0, 0), seasonal_order=(0, 0, 0, 0),
                  enforce_stationarity=False, enforce_invertibility=False)
fit = modelo.fit(method='lbfgs', maxiter=100, disp=False)

In [4]:
previsao = fit.get_forecast(steps=len(teste), exog=exog_teste)
media_prevista = previsao.predicted_mean

residuo_sarima = teste - media_prevista

In [5]:
with open('../data/labels/combined_windows.json') as f:
    labels = json.load(f)

windows = labels['realAWSCloudwatch/ec2_cpu_utilization_24ae8d.csv']
windows = [(pd.Timestamp(w[0]), pd.Timestamp(w[1])) for w in windows]

dentro_anomalia = pd.Series(False, index=teste.index)
for start, end in windows:
    dentro_anomalia |= (teste.index >= start) & (teste.index <= end)

In [6]:
JANELA = 24
sinal_sarima = residuo_sarima.abs().rolling(window=JANELA, min_periods=JANELA).std()

sinal_normal = sinal_sarima[~dentro_anomalia].dropna()
limiar_sarima = sinal_normal.quantile(0.99)

print(f'Limiar (P99 do período normal, resíduo SARIMA): {limiar_sarima:.4f}')

Limiar (P99 do período normal, resíduo SARIMA): 0.3037


In [7]:
previsao_anomalia_sarima = (sinal_sarima > limiar_sarima).fillna(False)

vp = (previsao_anomalia_sarima & dentro_anomalia).sum()
fp = (previsao_anomalia_sarima & ~dentro_anomalia).sum()
fn = (~previsao_anomalia_sarima & dentro_anomalia).sum()

precisao = vp / (vp + fp) if (vp + fp) > 0 else 0
recall = vp / (vp + fn) if (vp + fn) > 0 else 0
f1 = 2 * (precisao * recall) / (precisao + recall) if (precisao + recall) > 0 else 0

print(f'VP={vp}  FP={fp}  FN={fn}')
print(f'Precisão: {precisao:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}')

VP=24  FP=4  FN=378
Precisão: 0.857 | Recall: 0.060 | F1: 0.112


In [8]:
j2_inicio, j2_fim = windows[1]
sinal_j2 = sinal_sarima.loc[j2_inicio:j2_fim]

print(f'Máximo do sinal (SARIMA) dentro da Janela 2: {sinal_j2.max():.4f}')
print(f'Limiar atual: {limiar_sarima:.4f}')

Máximo do sinal (SARIMA) dentro da Janela 2: 0.1044
Limiar atual: 0.3037


In [9]:
comparacao_metodos = pd.DataFrame({
    'Decomposição simples': [0.923, 0.119, 0.211],   # valores que você obteve no notebook 07
    'SARIMA + Fourier': [precisao, recall, f1],
}, index=['Precisão', 'Recall', 'F1-score'])

comparacao_metodos

,Decomposição simples,SARIMA + Fourier
Precisão,0.923,0.857143
Recall,0.119,0.059701
F1-score,0.211,0.111628
